# Baysor segmentation — Stroke MERSCOPE (Fan / CG)

Batch-runs [Baysor](https://github.com/kharchenkolab/Baysor) cell segmentation on the raw MERSCOPE
transcript CSVs in `/Volumes/T7/Stroke_merscop_Fan_CG/transcript files`.

**Input format** (one CSV per region, MERSCOPE `detected_transcripts`-style):

| col | meaning |
|-----|---------|
| `global_x`, `global_y`, `global_z` | micron coordinates (z is the imaging plane index 0–6) |
| `gene` | gene / barcode name (includes `Blank_*` controls) |
| `fov`, `barcode_id`, `transcript_id`, `x`, `y` | metadata (unused) |

**What this notebook does**
1. Discovers every `*.csv` in the input folder (ignores macOS `._` resource-fork files).
2. For each region, runs Baysor with the lab-standard parameters (`scale=4`, `min_molecules_per_cell=50`,
   `n_clusters=3`, `iters=500`), excluding `Blank*` / control barcodes via `config.data.exclude_genes`
   so we never duplicate the (large) raw files.
3. Writes results to `…/baysor_segmentation/<sample>/m50_s4/segmentation*` and **skips** any region
   that already finished — so the loop is resumable if it is interrupted.

**Kernel:** `Baysor-16-Threads 1.10.9` (Julia 1.10, `JULIA_NUM_THREADS=16`).

> ⚠️ This is heavy: 51 regions, some files >1 GB. Expect this to run for many hours. It is safe to stop and
> re-run — finished regions are detected and skipped.

## 1. Environment

In [ ]:
using ProgressMeter
ProgressMeter.ijulia_behavior(:append)

import Pkg
Pkg.activate("/Users/christoffer/Baysor")
using Baysor
using DataFrames
using CSV

println("Julia threads: ", Threads.nthreads())   # should be 16 on the Baysor-16-Threads kernel

## 2. Configuration

In [ ]:
# --- paths -------------------------------------------------------------------
input_folder = "/Volumes/T7/Stroke_merscop_Fan_CG/transcript files"

# Outputs go to a sibling folder so the raw transcripts stay untouched.
output_root  = "/Volumes/T7/Stroke_merscop_Fan_CG/baysor_segmentation"
mkpath(output_root)

# --- Baysor parameters (lab standard for these mouse MERSCOPE panels) --------
baysor_params = (
    x_column = :global_x,
    y_column = :global_y,
    z_column = :global_z,        # set force_2d=true below to ignore z
    gene_column = :gene,
    min_molecules_per_cell = 50,
    n_clusters = 3,
    scale = 4.0,                 # expected cell radius, microns
    iters = 500,
)

# Control / blank barcodes to drop (unanchored regex, matched against gene names).
exclude_genes = "^Blank,^NegControl,^NegativeControl,^Unassigned,^None\$,^DeprecatedCodeword,^FalseCode"

# Folder name encoding the params, e.g. m50_s4
param_tag = "m$(baysor_params.min_molecules_per_cell)_s$(Int(baysor_params.scale))"
println("param_tag = ", param_tag)
println("exclude_genes = ", exclude_genes)

## 3. Discover input files

Lists every real `.csv` in the input folder, derives a sample name from the file name, and checks whether a
completed segmentation already exists (a non-empty `segmentation.csv` in the output folder).

In [ ]:
"""Output prefix for a given sample, e.g. <output_root>/<sample>/m50_s4/segmentation"""
output_prefix(sample::String) = joinpath(output_root, sample, param_tag, "segmentation")

"""A region counts as done if its segmentation.csv exists and is non-empty."""
function segmentation_done(sample::String)::Bool
    seg = output_prefix(sample) * ".csv"
    return isfile(seg) && filesize(seg) > 0
end

# Collect real CSVs (skip macOS AppleDouble `._` files and hidden dotfiles).
all_csvs = sort([f for f in readdir(input_folder)
                 if endswith(lowercase(f), ".csv") && !startswith(f, "._") && !startswith(f, ".")])

to_process = Tuple{String,String}[]   # (sample, full_path)
skipped    = String[]
for f in all_csvs
    sample = splitext(f)[1]
    path   = joinpath(input_folder, f)
    if segmentation_done(sample)
        push!(skipped, sample)
    else
        push!(to_process, (sample, path))
    end
end

println("="^80)
println("Found $(length(all_csvs)) CSV files")
println("  to process : $(length(to_process))")
println("  already done: $(length(skipped))")
println("="^80)
for (s, p) in to_process
    gb = round(filesize(p) / 1e9, digits=2)
    println("  [ ] $s  ($(gb) GB)")
end
for s in skipped
    println("  [x] $s  (done)")
end

## 4. Runner

In [ ]:
"""Run Baysor on one transcript CSV, writing to <output_root>/<sample>/<param_tag>/segmentation*"""
function run_baysor(sample::String, input_path::String; params...)
    out_prefix = output_prefix(sample)
    mkpath(dirname(out_prefix))

    println("\n" * "="^80)
    println("Running Baysor on: $sample")
    println("  input : $input_path")
    println("  output: $(out_prefix)*")
    println("="^80)

    # Build a config object so we can set iters + exclude_genes (not plain kwargs of run()).
    cfg = Baysor.Utils.RunOptions()
    cfg.segmentation.iters = Int(params[:iters])
    cfg.data.exclude_genes = exclude_genes
    # cfg.data.force_2d = true   # <- uncomment to segment in 2D (ignore global_z)

    Baysor.CommandLine.run(
        input_path;
        x_column = params[:x_column],
        y_column = params[:y_column],
        z_column = params[:z_column],
        gene_column = params[:gene_column],
        min_molecules_per_cell = params[:min_molecules_per_cell],
        n_clusters = params[:n_clusters],
        scale = params[:scale],
        output = out_prefix,
        config = cfg,
    )

    println("\nCompleted: $sample")
    return dirname(out_prefix)
end

## 5. Run the batch

Processes every pending region. Errors on one region are caught and logged so the rest still run; re-running
the notebook retries only the failed/unfinished ones.

In [ ]:
results = NamedTuple[]

for (i, (sample, path)) in enumerate(to_process)
    println("\n[$(i)/$(length(to_process))] $sample")
    try
        out = run_baysor(sample, path; baysor_params...)
        push!(results, (sample=sample, status="success", output=out))
    catch e
        println("\nERROR on $sample:")
        showerror(stdout, e); println()
        push!(results, (sample=sample, status="error", output=string(e)))
    end
end

n_ok  = count(r -> r.status == "success", results)
n_err = count(r -> r.status == "error", results)
println("\n" * "="^80)
println("BAYSOR BATCH COMPLETE")
println("  succeeded: $n_ok")
println("  errored  : $n_err")
println("  skipped (already done before this run): $(length(skipped))")
println("="^80)
for r in results
    r.status == "error" && println("  ERROR  $(r.sample): $(first(r.output, 200))")
end

## 6. Quick sanity check on outputs

In [ ]:
for f in sort(readdir(output_root))
    seg   = joinpath(output_root, f, param_tag, "segmentation.csv")
    stats = joinpath(output_root, f, param_tag, "segmentation_cell_stats.csv")
    if isfile(stats)
        n_cells = countlines(stats) - 1
        println(rpad(f, 40), "  cells = ", n_cells)
    elseif isfile(seg)
        println(rpad(f, 40), "  segmentation.csv present, stats missing")
    end
end